# Generate a MongoDB workload

**Goal:** generate patterns, save/load their instances, then constrain
selectivity and execute the resulting workload.

Start Docker and run all cells in order with the BONDI Python environment.
The first code cell creates the named `notebook-generation` deployment and loads the
eight fictional businesses into `demo.business`.
No external data or connection configuration is needed. If a port is occupied,
change the port arguments in the setup cell before the first run.

The final cell removes the deployment, including its data volume and saved configuration.
Rerunning setup creates a fresh deployment and reloads the demo documents. If you
interrupt execution, run the final cleanup cell manually. Query examples read the demo data; the setup cell writes it.

Companion guide: [workload generation](../docs/workload-generation.md).

In [ ]:
from bondi.deployment import deploy_mongodb

deployment = deploy_mongodb(name="notebook-generation", port=27123)
deployment.load_demo_data()


In [ ]:
from collections import Counter
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory

from bondi.mongodb import (
    MongoCollectionRef,
    MongoExecutor,
    MongoFindQuery,
    MongoGenerationConfig,
    MongoSampleEstimator,
    MongoWorkloadGenerator,
    summarize,
)
from bondi.schema import Schema
from bondi.workload import Workload

documents = [
    {"city": city, "stars": stars, "review_count": reviews}
    for city in ["Sydney", "Melbourne", "Brisbane"]
    for stars in [2.5, 3.0, 4.0, 4.5, 5.0]
    for reviews in [10, 50, 100]
]
schema = Schema.from_documents(documents, values="full")
target = MongoCollectionRef("demo", "business")
config = MongoGenerationConfig(min_predicates=1, max_predicates=2)
generator = MongoWorkloadGenerator(target=target, schema=schema, config=config)

## Ten patterns, fifty instances each
Full-retention schemas work too: frequencies are ignored, and values are sampled uniformly from distinct observed values. Returned pattern IDs index the pattern list.

In [ ]:
workload, patterns = generator.generate(
    num_patterns=10,
    instances_per_pattern=50,
    seed=42,
    return_patterns=True,
)
assert len(workload) == 500
print(Counter(entry.pattern_id for entry in workload))
entry = workload[0]
print(patterns[entry.pattern_id])
pprint(entry.query.filter)
assert all(patterns[e.pattern_id].matches(e.query) for e in workload)
pprint(generator.last_stats)

## Save, load and repeat
JSONL preserves queries and pattern IDs. Pattern shapes can be recovered from queries. Set instances_per_pattern=1 for one query per unique pattern.

In [ ]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "workload.jsonl"
    workload.save(path)
    restored = Workload.load(path)
    assert [(e.query.fingerprint(), e.pattern_id) for e in restored] == [
        (e.query.fingerprint(), e.pattern_id) for e in workload
    ]

repeated = generator.generate(num_patterns=10, instances_per_pattern=50, seed=42)
assert [e.query.fingerprint() for e in repeated] == [e.query.fingerprint() for e in workload]
print("Loaded concrete query type:", type(restored[0].query).__name__)
print("Repeated generation matches with fixed inputs and seed.")

## Selectivity constraints against the deployment

Read `demo.business`, create an owned temporary sample, and remove the sample on
context exit. Source documents remain unchanged. A four-document sample illustrates
the API; use a representative larger sample for research.


In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=10) as executor:
    documents = executor.run(
        MongoFindQuery(target, {}, limit=100), collect_results=True
    ).documents
    live_schema = Schema.from_documents(documents, values="distinct")
    live_config = MongoGenerationConfig(
        min_predicates=1,
        max_predicates=2,
        include_fields=["city", "stars", "review_count"],
    )
    with MongoSampleEstimator(executor, target, sample_size=4, persist=True) as estimator:
        constrained = MongoWorkloadGenerator(
            target=target,
            schema=live_schema,
            config=live_config,
            estimator=estimator,
        )
        live_workload, live_patterns = constrained.generate(
            num_patterns=10,
            instances_per_pattern=50,
            min_selectivity=0.25,
            max_selectivity=0.75,
            seed=42,
            return_patterns=True,
        )
        assert len(live_workload) == 500
        assert all(
            0.25 <= estimator.estimate(e.query).selectivity <= 0.75 for e in live_workload
        )
        pprint(constrained.last_stats)
    assert estimator.sample_target is None

Bounds apply to each instance **before skip/limit**, relative to the sample. They
are not full-population guarantees. Database sampling is not controlled by the
generator seed. Impossible constraints raise `WorkloadGenerationError` rather than
returning an incomplete workload; inspect generation statistics to understand rejections.

## Execute and profile the generated workload

Use `live_workload` from the preceding section: the toy schema above does not
describe the database. Execution and profiling are separate passes, not
interchangeable timings. Concurrency stays at one so queries do not compete
within each pass.


In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=10) as executor:
    results = executor.run_workload(live_workload)
    profiles = executor.profile_workload(live_workload)

assert all(r is not None and r.error is None for r in results + profiles)
for entry, result, profile in list(zip(live_workload, results, profiles))[:5]:
    summary = summarize(profile.raw)
    print(
        entry.pattern_id,
        "client ms:",
        round(result.elapsed_ms, 2),
        "keys examined:",
        summary.keys_examined,
    )
print("Executed and profiled", len(live_workload), "queries")

## Try next

Set `instances_per_pattern=1` to generate one instance per pattern. Change the
field list, predicate counts or selectivity bounds and inspect `last_stats`.
Use a representative schema and a larger sample for research, and save the final
workload as the durable input to experiments.

The [generation reference](../docs/reference/mongodb/generation.md) documents all
configuration options and custom pattern validation. Logical filters and aggregation
generation are not implemented.

## Remove the deployment

Remove the database container, data volume and saved configuration. All executors
and temporary samples have already been closed by their context managers.
Run the setup cell again before continuing with database examples.


In [ ]:
deployment.remove(delete_data=True)
print("Deployment and data removed.")